# Part C — forecast lead-time signals

Compare the same valid rainfall day across successive ECMWF forecast starts for Khartoum, Kassala, Gedaref, El Obeid and Nyala. The analysis uses June 9–22, July 9–22 and August 9–22, 2025 (42 starts total), with each town’s local CMORPH 2-year, 24-hour threshold.

The source layout indicates about 7.5 GiB of decoded forecast chunks for a fresh runtime. Existing Part B point caches reduce this only if you have separately placed them in the runtime. The exact metadata-only estimate is printed before extraction; it does not measure compressed transfer or peak memory. Use a Kaggle or Colab runtime with internet enabled. Do not proceed with extraction if the printed budget or runtime is unsuitable.

Part A results are intentionally not committed. Provide these four files individually from the completed Part A output directory: `thresholds.csv`, `annual_maxima.csv`, `settings_used.json` and `manifest.json`. Do not upload a zip. In Kaggle, attach a dataset containing the files; in Colab, upload the files in the upload cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/BrianMburu/acmad-eac-summer-school.git'
BRANCH = 'forecast-lead-time'
REPO = Path('/kaggle/working/acmad-eac-summer-school') if Path('/kaggle/working').exists() else Path('/content/acmad-eac-summer-school')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Repository:', REPO, '| branch:', BRANCH)

In [ ]:
%pip install -q 'icechunk>=2.2.2,<3' 'zarr>=3.2,<4' 'xarray>=2025.1' 'dask[array]' numpy pandas scipy matplotlib 'xclim>=0.59,<1'

## Part A handoff

For Kaggle, set `A_RESULTS_DIR` to the attached dataset folder. For Colab, run the upload cell and select the four files individually. The code verifies their checksums and compatibility before using the thresholds.

In [ ]:
REQUIRED = ['thresholds.csv', 'annual_maxima.csv', 'settings_used.json', 'manifest.json']
KAGGLE_HANDOFF = Path('/kaggle/input/part-a-handoff')
A_RESULTS_DIR = KAGGLE_HANDOFF if KAGGLE_HANDOFF.exists() else Path('/content/part_a_handoff')
if not all((A_RESULTS_DIR / name).is_file() for name in REQUIRED) and Path('/content').exists():
    from google.colab import files
    uploaded = files.upload()  # select the four files individually; no archive needed
    A_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    for name in REQUIRED:
        if name in uploaded:
            (A_RESULTS_DIR / name).write_bytes(uploaded[name])
missing = [name for name in REQUIRED if not (A_RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing {missing} in {A_RESULTS_DIR}. Attach/upload all four Part A files individually.')
print('Verified handoff files found in:', A_RESULTS_DIR)

## 1. Inspect the source read

This stage opens forecast metadata and checks the selected towns, dates, available starts, existing Part B point caches and decoded source-chunk budget. It fetches no rainfall values. Review the estimate and warning before running the next cell.

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.forecast_lead_time', 'inspect', '--thresholds', str(A_RESULTS_DIR)], check=True)

## 2. Extract and analyze

Run this only after reviewing the inspection estimate. The script processes one initialization at a time, saves validated point caches so interrupted runs can resume, and records unavailable starts rather than treating them as dry. This is the heavy network and computation step.

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.forecast_lead_time', 'extract', '--thresholds', str(A_RESULTS_DIR)], check=True)

## 3. Create figures and inspect outputs

The offline plotting stage creates one forecast-start versus lead-day heatmap and one plume for the strongest sampled start at each town. Outputs are saved under `out/forecast_lead_time/`; the notebook runtime’s working directory can be downloaded or retained as notebook output.

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.forecast_lead_time', 'plots'], check=True)
import pandas as pd
out = REPO / 'out' / 'forecast_lead_time'
display(pd.read_csv(out / 'lead_time_summary.csv'))
display(pd.read_csv(out / 'most_alarming_starts.csv'))
print('Tables, manifest and figures:', out)